# 01 — What is machine learning?

*Module 00 · Getting Started — notebook 1 of 11*

Welcome. This is where it all begins, and it begins gently: by the end you'll be able to say what machine learning actually *is*, in your own words, with a real dataset in front of you.

**Prerequisites:** none — this is the entry point.

**What you'll be able to do**
- Explain what "learning a rule from examples" means.
- Point to the **features** and the **label** in a dataset, and see why one row is one example.
- Tell **classification** from **regression**.
- Read the scatter plot we'll return to in every notebook of this module.

In [ ]:
# Standard library
import sys

# Third-party
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

# The course toolkit
from ml_course import colors, datasets, viz

# A fixed seed makes everything reproducible: the same code gives the same result every time.
np.random.seed(0)

# Apply the course's visual style once, here, so every figure looks consistent.
viz.use_course_style()

# The two species we work with throughout this module, in a fixed order.
SPECIES_ORDER = ["Adelie", "Gentoo"]

# Print the exact versions we run on, so a result is always traceable to its tools.
print("python      ", sys.version.split()[0])
print("numpy       ", np.__version__)
print("pandas      ", pd.__version__)
print("scikit-learn", sklearn.__version__)
print("matplotlib  ", matplotlib.__version__)

## From writing rules to learning them

Suppose someone hands you a pile of measurements about penguins and asks you to tell two species apart. One way is to *write the rule yourself*: "if the flipper is longer than 205 mm, call it a Gentoo." You would study the data, guess a threshold, test it, adjust.

Machine learning takes a different route. Instead of writing the rule by hand, we let an algorithm **find the rule from labelled examples** — penguins whose species we already know. We show it many examples, and it works out a rule that separates them. That shift — from *coding the rule* to *learning the rule from data* — is the whole idea. Everything else in this course is a different way of doing that learning.

## Learning from labelled examples

The setting we'll live in for most of this course is **supervised learning**: every example comes with the answer attached.

- Each example has some **features** — the measurements we get to look at (here: a penguin's bill length and flipper length).
- Each example also has a **label** — the answer we want to predict (here: the species).

A **model** is a rule that takes the features of a new example and returns a predicted label. *Training* a model means letting it adjust that rule using examples whose labels we already know. We build our very first model by hand in notebook 05; for now, we get to know the raw material.

In [ ]:
df = datasets.load_penguins()

print(f"{len(df)} penguins in our dataset\n")
print("How many of each species:")
print(df["species"].value_counts().to_string())

df.head()

### Read the table

Each **row** is one penguin — one **example**. The two numeric columns, `bill_length_mm` and `flipper_length_mm`, are the **features**: what we get to measure. The `species` column is the **label**: the answer we'd like to predict from those features (its values are the strings `Adelie` and `Gentoo`).

So our question, stated precisely, is: *given a penguin's bill length and flipper length, can we predict its species?* These 274 penguins are a deliberately simple slice — two species, two measurements — chosen so we can see everything in a single picture; we take a closer look at this dataset in notebook 03 before we build anything.

## Two kinds of prediction

The label decides what kind of task we have.

- When the label is a **category** — like a species, `Adelie` or `Gentoo` — predicting it is **classification**.
- When the label is a **number** — like a penguin's body mass in grams — predicting it is **regression**.

This course is about **classification**: every one of the twelve methods we'll meet can sort examples into categories. Our penguins, with a species label, are a classification problem.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))

for species, color in zip(SPECIES_ORDER, colors.CLASS_CYCLE, strict=False):
    sub = df[df["species"] == species]
    ax.scatter(
        sub["bill_length_mm"],
        sub["flipper_length_mm"],
        color=color,
        edgecolor=colors.COLORS["text"],
        linewidth=0.5,
        s=40,
        label=species,
    )

ax.set_xlabel("bill_length_mm")
ax.set_ylabel("flipper_length_mm")
ax.set_title("Two penguin species in feature space")
ax.legend(title="species")
plt.show()

### Read the figure

Every penguin is a single point. Its horizontal position is its bill length, its vertical position its flipper length — the two axes are our two features. Colour shows the true species.

The two species sit in **clearly distinct regions** with a visible gap between them: Gentoos toward longer flippers and larger bills, Adélies toward shorter flippers and smaller bills. That separation is why a rule based on these two measurements has a chance of working — a single well-placed straight line would already get nearly every penguin right. What it could not do is get *every* one right: a few penguins sit close to the border, and neither measurement on its own fully separates the species. Mostly separable, but not perfectly — that is the honest starting point for everything that follows.

## Where would you draw the line?

Look again at the scatter and imagine drawing a single line to separate the two colours. You could place a good one by eye — a rule that gets most penguins right. That line *is* a rule: everything on one side is predicted Gentoo, everything on the other Adélie.

The work of machine learning is to find such a rule from the examples — and to do it in a way that holds up on penguins we have not seen yet. We build our first such rule, by hand, in notebook 05. First, let's see what it means to *use* a rule.

In [ ]:
# A new penguin we measured, but whose species we do not know.
new_penguin = pd.Series({"bill_length_mm": 48.0, "flipper_length_mm": 216.0})

fig, ax = plt.subplots(figsize=(7, 5.5))

for species, color in zip(SPECIES_ORDER, colors.CLASS_CYCLE, strict=False):
    sub = df[df["species"] == species]
    ax.scatter(
        sub["bill_length_mm"],
        sub["flipper_length_mm"],
        color=color,
        edgecolor=colors.COLORS["text"],
        linewidth=0.5,
        s=40,
        label=species,
        alpha=0.6,
    )

ax.scatter(
    new_penguin["bill_length_mm"],
    new_penguin["flipper_length_mm"],
    color=colors.COLORS["highlight"],
    edgecolor=colors.COLORS["text"],
    s=200,
    marker="*",
    label="new penguin (?)",
    zorder=5,
)

ax.set_xlabel("bill_length_mm")
ax.set_ylabel("flipper_length_mm")
ax.set_title("Which species is the new penguin?")
ax.legend(title="species")
plt.show()

### Read the figure

The star is the new penguin: measured, but unlabelled. To guess its species, we ask the same question you answered by eye — which cloud does it fall into? It lands among the longer-flippered points (the Gentoos), so that is our prediction.

That move — taking a new example and returning a label — is exactly what a trained model does. The model's whole job is to make that guess automatically, and well, for examples it has never seen.

## The words we'll keep using

We'll grow this list as the module goes. So far:

- **Example** — one row of data (one penguin).
- **Feature** — an input measurement we get to look at (bill length, flipper length).
- **Label** — the answer we want to predict (species).
- **Supervised learning** — learning from examples whose labels we already know.
- **Classification** — predicting a category. **Regression** — predicting a number.
- **Model** — a rule from features to a predicted label.
- **Prediction** — applying that rule to a new example.

## Your turn

1. In the penguins table, name the two features and the label, and say how many examples (rows) the dataset has. (You printed the count above.)
2. For each task, say whether it is classification or regression:
   - predicting a penguin's species from its measurements;
   - predicting a penguin's body mass in grams;
   - predicting which island a penguin lives on.
3. Looking at the scatter, describe in one sentence a rule (a line) that separates the two species — and name one region where your rule would get a penguin wrong.

These are yours to think through. If you'd like, write your answers in a new cell below.

## What you built

In one notebook, you've gone from "machine learning" as a vague phrase to something concrete:

- You can say what it means to **learn a rule from examples**, rather than writing the rule by hand.
- You can point to the **features** and the **label** in a real dataset, and you know one row is one example.
- You can tell **classification** (predicting a category) from **regression** (predicting a number).
- You can read the **feature-space scatter** — and you've already made a prediction by eye.

That last skill — reading the picture before trusting any number — is the habit this whole course is built on. Well done.

## Going further (optional)

Supervised learning is one of three classic settings, and it's the one this course focuses on:

- **Supervised** — examples come with labels (our case).
- **Unsupervised** — examples have no labels; the goal is to find structure (for example, grouping similar penguins without being told the species).
- **Reinforcement** — an agent learns by acting and receiving rewards (for example, learning to play a game).

You don't need these distinctions for the next notebook — they're here for the curious.

## References

- A. Géron, *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*, 3rd ed., O'Reilly, 2022 — chapter 1 (the machine-learning landscape).
- G. James, D. Witten, T. Hastie, R. Tibshirani, *An Introduction to Statistical Learning*, 2nd ed., Springer, 2021 — §2.1. DOI: 10.1007/978-1-0716-1418-1
- Data: K. B. Gorman, T. D. Williams, W. R. Fraser (2014), *Ecological sexual dimorphism and environmental variability within a community of Antarctic penguins (genus Pygoscelis)*, PLoS ONE 9(3):e90081. DOI: 10.1371/journal.pone.0090081. Packaged as palmerpenguins (A. M. Horst, A. P. Hill, K. B. Gorman, 2020), DOI: 10.5281/zenodo.3960218.

---
Previous: — · Next: **02 — Features, labels, and the feature space**